# 02 — Preprocessing & Feature Engineering
## Diabetes Risk Prediction · Clinical ML Portfolio

**Inputs:** `data/diabetes_nulls_flagged.csv`  
**Outputs:** `data/X_train.csv`, `data/X_test.csv`, `data/y_train.csv`, `data/y_test.csv`

This notebook handles: missing value imputation, outlier capping, feature scaling, train/test split, and final data validation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

os.makedirs('../data', exist_ok=True)
os.makedirs('../src', exist_ok=True)

print('Libraries loaded OK')

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/diabetes_nulls_flagged.csv')

X = df.drop('Outcome', axis=1)
y = df['Outcome']

print(f'Features shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'\nMissing values:\n{X.isnull().sum()}')

## 2. Train/Test Split

Split **before** imputation to prevent data leakage. Imputation statistics (medians) are computed on training data only and then applied to test data — same as in real deployment.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # preserves class ratio in both splits
)

print(f'Train set: {X_train.shape[0]} samples ({y_train.mean()*100:.1f}% positive)')
print(f'Test set:  {X_test.shape[0]} samples ({y_test.mean()*100:.1f}% positive)')
print('\n→ Stratification confirmed: class ratio preserved in both splits.')

## 3. Missing Value Imputation

**Strategy: median imputation stratified by outcome.**

Using the overall median would blend the distributions of diabetic and non-diabetic patients. Stratified imputation — using the median of each class separately — is more clinically coherent: a missing glucose in a diabetic patient is more likely to be high than the overall median.

In [ ]:
def stratified_median_impute(X_train, y_train, X_test):
    """
    Impute missing values using median computed per class on training data.
    Applied to both train and test sets using train-derived medians only.
    """
    X_train_imp = X_train.copy()
    X_test_imp = X_test.copy()
    
    imputation_stats = {}
    
    for col in X_train.columns:
        if X_train[col].isnull().sum() > 0:
            # Compute medians from training data only
            median_0 = X_train[col][y_train == 0].median()
            median_1 = X_train[col][y_train == 1].median()
            overall_median = X_train[col].median()
            
            imputation_stats[col] = {
                'median_no_diabetes': median_0,
                'median_diabetes': median_1,
                'overall_median': overall_median
            }
            
            # Impute train set (stratified)
            mask_0 = X_train_imp[col].isnull() & (y_train == 0)
            mask_1 = X_train_imp[col].isnull() & (y_train == 1)
            X_train_imp.loc[mask_0, col] = median_0
            X_train_imp.loc[mask_1, col] = median_1
            
            # Impute test set (overall median — outcome not known at inference)
            X_test_imp[col] = X_test_imp[col].fillna(overall_median)
    
    return X_train_imp, X_test_imp, imputation_stats


X_train_imp, X_test_imp, imputation_stats = stratified_median_impute(X_train, y_train, X_test)

print('Imputation statistics (training medians):')
print(f'{"Feature":<28} {"No DM median":>14} {"DM median":>12} {"Overall":>10}')
print('-' * 68)
for col, stats_dict in imputation_stats.items():
    print(f'{col:<28} {stats_dict["median_no_diabetes"]:>14.2f} {stats_dict["median_diabetes"]:>12.2f} {stats_dict["overall_median"]:>10.2f}')

print(f'\nMissing after imputation — Train: {X_train_imp.isnull().sum().sum()}, Test: {X_test_imp.isnull().sum().sum()}')

# Save imputation stats for app deployment
joblib.dump(imputation_stats, '../src/imputation_stats.pkl')
print('Saved: src/imputation_stats.pkl')

## 4. Outlier Capping

Cap extreme values at the 99th percentile (computed on training data). Preserves clinical variation while reducing the influence of physiologically implausible extremes on model training.

In [ ]:
cap_features = ['Insulin', 'SkinThickness', 'BloodPressure']
cap_values = {}

for col in cap_features:
    upper = X_train_imp[col].quantile(0.99)
    lower = X_train_imp[col].quantile(0.01)
    cap_values[col] = {'lower': lower, 'upper': upper}
    
    n_capped_train = ((X_train_imp[col] > upper) | (X_train_imp[col] < lower)).sum()
    
    X_train_imp[col] = X_train_imp[col].clip(lower, upper)
    X_test_imp[col] = X_test_imp[col].clip(lower, upper)
    
    print(f'{col}: capped at [{lower:.1f}, {upper:.1f}] — {n_capped_train} training values affected')

joblib.dump(cap_values, '../src/cap_values.pkl')
print('\nSaved: src/cap_values.pkl')

## 5. Feature Scaling

StandardScaler fitted on training data only. Required for Logistic Regression and SVM. Random Forest is scale-invariant, but we scale all models for consistency.

In [ ]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_imp),
    columns=X_train_imp.columns,
    index=X_train_imp.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_imp),
    columns=X_test_imp.columns,
    index=X_test_imp.index
)

print('Scaling applied. Feature statistics after scaling (train):')
print(X_train_scaled.describe().round(3).loc[['mean', 'std']])

# Save scaler
joblib.dump(scaler, '../src/scaler.pkl')
print('\nSaved: src/scaler.pkl')

## 6. Save Processed Datasets

In [ ]:
# Save both scaled (for LR/SVM) and unscaled (for RF)
X_train_scaled.to_csv('../data/X_train_scaled.csv', index=False)
X_test_scaled.to_csv('../data/X_test_scaled.csv', index=False)
X_train_imp.to_csv('../data/X_train.csv', index=False)
X_test_imp.to_csv('../data/X_test.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)

print('All datasets saved:')
for f in ['X_train.csv', 'X_test.csv', 'X_train_scaled.csv', 'X_test_scaled.csv', 'y_train.csv', 'y_test.csv']:
    print(f'  data/{f}')

print(f'\nFinal train shape: {X_train_imp.shape}')
print(f'Final test shape:  {X_test_imp.shape}')
print('\n→ Proceed to notebook 03: Model Training & Evaluation')